# Архитектура BERT и Fine-tuning

__Автор задач: Блохин Н.В. (NVBlokhin@fa.ru)__

Материалы:
* https://huggingface.co/docs/transformers/index
* https://huggingface.co/docs/transformers/main_classes/pipelines#transformers.pipeline.task
* https://huggingface.co/docs/transformers/preprocessing
* https://huggingface.co/blog/getting-started-with-embeddings
----
* https://huggingface.co/docs/datasets/main/en/repository_structure
* https://huggingface.co/docs/datasets/main/en/package_reference/loading_methods#datasets.load_dataset
* https://huggingface.co/docs/datasets/process
----
* https://huggingface.co/docs/transformers/training
* https://huggingface.co/docs/transformers/main_classes/trainer
* https://huggingface.co/docs/transformers/main_classes/trainer#transformers.TrainingArguments
* https://huggingface.co/learn/llm-course/chapter3/3
* https://huggingface.co/learn/llm-course/chapter3/4
---
* https://huggingface.co/docs/evaluate/index
* https://huggingface.co/sentence-transformers

In [1]:
import os
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from datasets import Dataset, load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModel, 
    AutoModelForMaskedLM,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EvalPrediction
)
from sentence_transformers import SentenceTransformer, util


if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
    
print(f"Используемое устройство: {device}")


# Создадим директории, если их нет
os.makedirs("data", exist_ok=True)
os.makedirs("reviews_polarity_dataset", exist_ok=True)

Используемое устройство: cuda


## Задачи для совместного разбора

1\. Обсудите основные возможности и экосистему пакета 🤗 Transformers на примере задачи поиска ответа на вопрос в тексте.

In [2]:
text = """The seminars on Deep Learning and Natural Language Processing were truly captivating,
providing a deep dive into the intricacies of these disciplines.
The wealth of knowledge and insights gained during the sessions was commendable.
However, it's disheartening to note the scarcity of homework assignments.
Anastasia, in particular, is quite concerned that the limited number of assignments might
fall short of even reaching 30. While the seminars were intellectually stimulating,
the desire for more hands-on practice through assignments remains strong,
as it is crucial for reinforcing the theoretical understanding acquired during the classes."""

In [3]:
question1 = "What would be the ideal number of homework assignments for Anastasia"
question2 = "What are the shortcomings of the course?"

In [4]:
from transformers import pipeline

In [5]:
# answerer

In [6]:
# answerer = pipeline(
#     "document-question-answering",
#     model="distilbert/distilbert-base-cased-distilled-squad",
# )
# # answerer(question=question1, context=text)

In [7]:
# from transformers import DistilBertTokenizer, DistilBertModel
# import torch

# tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-cased-distilled-squad')
# model = DistilBertModel.from_pretrained('distilbert-base-cased-distilled-squad')

# question, text = "Who was Jim Henson?", "Jim Henson was a nice puppet"

# inputs = tokenizer(question, text, return_tensors="pt")
# with torch.no_grad():
#     outputs = model(**inputs)

# print(outputs)


In [8]:
# model

2\. Обсудите основные шаги по дообучению моделей из экосистемы 🤗 Transformers.

In [9]:
# from transformers import TrainingArguments



In [10]:
# TrainingArguments()

## Задачи для самостоятельного решения

<p class="task" id="1"></p>

1\. Загрузите модель `cointegrated/rubert-tiny2` (или любую другую небольшую модель BERT) и соответствующий токенизатор из библиотеки `transformers`.

Придумайте два предложения с омонимами. Токенизируйте оба предложения. Прогоните их через модель. Найдите индексы токенов, соответствующих ономиничному слову в каждом предложении. Извлеките их эмбеддинги с последнего скрытого слоя модели. Посчитайте косинусное сходство между этими двумя векторами. Выведите и прокомментируйте результат.

- [ ] Проверено на семинаре


In [3]:
model_name = "cointegrated/rubert-tiny2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)


model.eval()

text1 = "Моя бабуля делает очень вкусный суп. Его ингредиенты - лук и чеснок. Гойда"
text2 = "Остроухий эльф натянул свой верный лук и прицелился в орка. Гойда"
word_to_find = "Гой"

def get_word_embedding(text: str, word: str) -> torch.Tensor:
    inputs = tokenizer(text, return_tensors="pt").to(device)
    
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    
    try:
        word_idx = tokens.index(word)
    except ValueError:
        raise ValueError(f"Слово '{word}' (или такой токен) не найдено в тексте: {tokens}")
        
    print(f"Токены: {tokens}")
    print(f"Индекс слова '{word}': {word_idx}\n")
    
    with torch.no_grad():
        outputs = model(**inputs)
        
    embedding = outputs.last_hidden_state[0, word_idx, :]
    return embedding

emb1 = get_word_embedding(text1, word_to_find)

emb2 = get_word_embedding(text2, word_to_find)


cos_sim = F.cosine_similarity(emb1.unsqueeze(0), emb2.unsqueeze(0))

print(f"Косинусное сходство между эмбеддингами слова '{word_to_find}': {cos_sim.item():.4f}")

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Токены: ['[CLS]', 'Моя', 'бабу', '##ля', 'делает', 'очень', 'вкусный', 'суп', '.', 'Его', 'ингредиенты', '-', 'лук', 'и', 'чеснок', '.', 'Гой', '##да', '[SEP]']
Индекс слова 'Гой': 16

Токены: ['[CLS]', 'Остр', '##оу', '##хий', 'эльф', 'натя', '##нул', 'свой', 'верный', 'лук', 'и', 'прице', '##лился', 'в', 'ор', '##ка', '.', 'Гой', '##да', '[SEP]']
Индекс слова 'Гой': 17

Косинусное сходство между эмбеддингами слова 'Гой': 0.9335


<p class="task" id="2"></p>

2\. Загрузите модель с головой для языкового моделирования `AutoModelForMaskedLM` из библиотеки `transformers` (используйте ту же модель `cointegrated/rubert-tiny2` или `bert-base-uncased` для английских примеров).

Создайте функцию `predict_mask`, которая:
- принимает предложение, содержащее специальный токен [MASK].
- токенизирует текст и пропускает через модель.
- находит индекс токена маски.
- возвращает топ-5 слов с наибольшей вероятностью для этой позиции.

Придумайте и протестируйте по одному примеру на каждый из следующих сценариев:
- Составьте предложение-факт, где маской скрыт объект, географическое название или имя, которое модель должна помнить из этапа предобучения. Пример: "Столица Франции — это [MASK]."
- Придумайте шаблон предложения, в котором изменение одного контекстного слова (например, цели действия) кардинально меняет смысл пропущенного слова. Прогоните шаблон дважды с разным контекстом. Пример: "Я пошел в [MASK], чтобы купить хлеб" и "Я пошел в [MASK], чтобы купить акции".
- Составьте предложение с упоминанием профессии или социальной роли, скрыв маской местоимение (так, чтобы из контекста не был понятен пол). Посмотрите, какие местоимения он предложит. Пример: "Доктору пришлось бежать, потому что [MASK] внезапно позвали"


- [ ] Проверено на семинаре

In [12]:
mlm_model = AutoModelForMaskedLM.from_pretrained(model_name).to(device)
mlm_model.eval()

def predict_mask(text: str, top_k: int = 5):
    if tokenizer.mask_token not in text:
        raise ValueError(f"Текст должен содержать токен {tokenizer.mask_token}")
        
    inputs = tokenizer(text, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = mlm_model(**inputs)
    
    logits = outputs.logits
    
    mask_token_index = torch.where(inputs.input_ids == tokenizer.mask_token_id)[1][0]
    
    mask_logits = logits[0, mask_token_index, :]
    
    top_tokens = torch.topk(mask_logits, top_k).indices
    
    print(f"\nТекст: {text}")
    print("Топ-5 предсказаний:")
    for i, token_id in enumerate(top_tokens):
        word = tokenizer.decode([token_id])
        print(f"{i+1}. {word}")

predict_mask(f"Столица России — это {tokenizer.mask_token}.")

predict_mask(f"Я пошел в {tokenizer.mask_token}, чтобы получить новые знания.")
predict_mask(f"Я пошел в {tokenizer.mask_token}, чтобы поесть блинчиков с курицей.")

predict_mask(f"{tokenizer.mask_token} персонаж был плохо прописан автором")

Loading weights:   0%|          | 0/58 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: cointegrated/rubert-tiny2
Key                          | Status     |  | 
-----------------------------+------------+--+-
cls.seq_relationship.weight  | UNEXPECTED |  | 
bert.pooler.dense.weight     | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 
bert.pooler.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Текст: Столица России — это [MASK].
Топ-5 предсказаний:
1. страна
2. Россия
3. история
4. город
5. государство

Текст: Я пошел в [MASK], чтобы получить новые знания.
Топ-5 предсказаний:
1. школу
2. центр
3. путь
4. университет
5. Москву

Текст: Я пошел в [MASK], чтобы поесть блинчиков с курицей.
Топ-5 предсказаний:
1. школу
2. магазин
3. лес
4. туалет
5. центр

Текст: [MASK] персонаж был плохо прописан автором
Топ-5 предсказаний:
1. Этот
2. Его
3. Данный
4. Такой
5. Главный


<p class="task" id="3"></p>

3\. Разбейте данные из файла `reviews_polarity.csv` на обучающее и валидационное множество в соотношении 80 на 20. Создайте папку `reviews_polarity_dataset` и сохраните в нее полученные фрагменты данных под названием `train.csv` и `test.csv`. Создайте объект `datasets.Dataset`, используя функцию `load_dataset`.

Токенизируйте строки при помощи токенизатора, соотвествующего модели `rubert-base-cased-sentiment`. Удалите из датасета поле `text` после токенизации, замените поле `class` на `labels` и приведите данные к тензорам `torch`.

Создайте два `DataLoader` на основе обучающего и валидационного множества. Получите батч из обучающего множества и выведите его на экран.

In [13]:
file_path = 'data/reviews_polarity.csv'

df = pd.read_csv(file_path)
dataset_dir = "reviews_polarity_dataset"
os.makedirs(dataset_dir, exist_ok=True)

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['class'])
train_df.to_csv(f"{dataset_dir}/train.csv", index=False)
test_df.to_csv(f"{dataset_dir}/test.csv", index=False)

raw_datasets = load_dataset(
    "csv", 
    data_files={"train": f"{dataset_dir}/train.csv", "validation": f"{dataset_dir}/test.csv"}
)


task3_tokenizer = AutoTokenizer.from_pretrained("blanchefort/rubert-base-cased-sentiment")

def tokenize_function(examples):
    return task3_tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

tokenized_datasets = tokenized_datasets.rename_column("class", "labels")
tokenized_datasets = tokenized_datasets.remove_columns(["text"])

tokenized_datasets.set_format("torch")

print("\nСтруктура датасета после обработки:")
print(tokenized_datasets)



train_dataloader = DataLoader(tokenized_datasets["train"], shuffle=True, batch_size=8)
eval_dataloader = DataLoader(tokenized_datasets["validation"], batch_size=8)

batch = next(iter(train_dataloader))
print("\nКлючи в батче:", batch.keys())
print("Размерность input_ids (Токены):", batch["input_ids"].shape) # Ожидаем [Batch, Seq_Len]
print("Размерность labels (Метки классов):", batch["labels"].shape)   # Ожидаем[Batch]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/30574 [00:00<?, ? examples/s]

Map:   0%|          | 0/7644 [00:00<?, ? examples/s]


Структура датасета после обработки:
DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 30574
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 7644
    })
})

Ключи в батче: dict_keys(['labels', 'input_ids', 'token_type_ids', 'attention_mask'])
Размерность input_ids (Токены): torch.Size([8, 128])
Размерность labels (Метки классов): torch.Size([8])


<p class="task" id="4"></p>

4\. Создайте модель при помощи класса `AutoModelForSequenceClassification`, заменив голову модели в соответствии с задачей бинарной классификации. Используя `transformers.Trainer`, настройте модель для решения задачи бинарной классификации. При настройке `Trainer` укажите количество эпох, равное 5. Во время обучения выводите на экран значение функции потерь на обучающем множестве и f1 на валидационном множестве.  

- [ ] Проверено на семинаре


In [13]:
MODEL_NAME = "blanchefort/rubert-base-cased-sentiment"

classifier_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True
).to(device)

def compute_metrics(p: EvalPrediction):
    preds = np.argmax(p.predictions, axis=1)
    f1 = f1_score(p.label_ids, preds, average="weighted")
    return {"f1": f1}

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch", 
    logging_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    load_best_model_at_end=True,
    report_to="none"
)


trainer = Trainer(
    model=classifier_model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,
)

trainer.train()

You passed `num_labels=2` which is incompatible to the `id2label` map of length `3`.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: blanchefort/rubert-base-cased-sentiment
Key                          | Status     |                                                                                       
-----------------------------+------------+---------------------------------------------------------------------------------------
bert.embeddings.position_ids | UNEXPECTED |                                                                                       
classifier.bias              | MISMATCH   | Reinit due to size mismatch - ckpt: torch.Size([3]) vs model:torch.Size([2])          
classifier.weight            | MISMATCH   | Reinit due to size mismatch - ckpt: torch.Size([3, 768]) vs model:torch.Size([2, 768])

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Запускаем процесс дообучения (Fine-tuning)...


Epoch,Training Loss,Validation Loss,F1
1,0.526444,0.521316,0.692866
2,0.521359,0.518730,0.692866
3,0.520578,0.519243,0.692866
4,0.519997,0.521025,0.692866
5,0.520242,0.520419,0.692866


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=19110, training_loss=0.5217239547561813, metrics={'train_runtime': 1579.0506, 'train_samples_per_second': 96.811, 'train_steps_per_second': 12.102, 'total_flos': 1.00554467582208e+16, 'train_loss': 0.5217239547561813, 'epoch': 5.0})

In [14]:
MODEL_NAME = "cointegrated/rubert-tiny2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
classifier_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=2
).to(device)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.rename_column("class", "labels")
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets.set_format("torch")

def compute_metrics(p: EvalPrediction):
    preds = np.argmax(p.predictions, axis=1)
    f1 = f1_score(p.label_ids, preds, average="weighted")
    return {"f1": f1}

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    load_best_model_at_end=True,
    report_to="none" 
)

trainer = Trainer(
    model=classifier_model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,
)

trainer.train()

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider trai

Map:   0%|          | 0/30574 [00:00<?, ? examples/s]

Map:   0%|          | 0/7644 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,F1
1,0.320425,0.292075,0.887473
2,0.278579,0.289649,0.892575
3,0.258347,0.319181,0.895563
4,0.242341,0.351999,0.892307
5,0.233179,0.369846,0.892804


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.beta', 'bert.embeddings.LayerNorm.gamma', 'bert.encoder.layer.0.attention.output.LayerNorm.beta', 'bert.encoder.layer.0.attention.output.LayerNorm.gamma', 'bert.

TrainOutput(global_step=19110, training_loss=0.26657418770543073, metrics={'train_runtime': 290.3893, 'train_samples_per_second': 526.431, 'train_steps_per_second': 65.808, 'total_flos': 281823755105280.0, 'train_loss': 0.26657418770543073, 'epoch': 5.0})

<p class="task" id="5"></p>

5\. Используя эмбеддинги `distiluse-base-multilingual-cased-v1` из пакета `sentence_transformers`, решите поиска семантически близких заголовков. Для этого закодируйте список текстов заголовков, посчитайте косинусную близость между одним выбранным заголовком и всеми остальными и выведите на экран k наиболее похожих. Для можете воспользоваться функцией `semantic_search` из пакета `sentence_transformers`.

- [ ] Проверено на семинаре

In [11]:
sbert_model = SentenceTransformer('distiluse-base-multilingual-cased-v1')

headlines = df.text.values

query_idx = int(torch.randint(0, len(headlines)-1,(1,))[0])
query_text = headlines[query_idx]
print(f"Наш запрос (Query): '{query_text}'\n")

corpus = [h for i, h in enumerate(headlines) if i != query_idx]


query_embedding = sbert_model.encode(query_text, convert_to_tensor=True)
corpus_embeddings = sbert_model.encode(corpus, convert_to_tensor=True)

top_k = 5
hits = util.semantic_search(query_embedding, corpus_embeddings, top_k=top_k)


print(f"Топ-{top_k} самых близких по смыслу заголовков:")
for hit in hits[0]:
    score = hit['score']
    matched_idx = hit['corpus_id']
    print(f"Скор: {score:.4f} | Заголовок: '{corpus[matched_idx]}'")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Наш запрос (Query): 'Ценники не соответствуют цене на кассе или вовсе отсутствуют, персонал хамит и ничего не знает (включая заведующую), много просрочки.'

Топ-5 самых близких по смыслу заголовков:
Скор: 0.6897 | Заголовок: 'Неприветливый персонал, цены на кассе не соответствуют ценам на полочках.'
Скор: 0.6128 | Заголовок: 'Ценников либо нет... либо не соответствуют... кассы одна зато персонал стоит и общается где нибудь в отделе....'
Скор: 0.6126 | Заголовок: 'Ценники либо стоят не на тот товар либо вообще отсутствуют да и персонал грубый хамливый мативируют это тем что они работают без выходных и вечные очереди на кассу'
Скор: 0.5866 | Заголовок: 'Товар есть, цены нет. На кассе не знают код. Есть ценник ...нет товара. Персонал хамит покупателям. Жуткий магазин. '
Скор: 0.5832 | Заголовок: 'Ценники не соответствуют цене товара, часто узнаешь только на кассе. Мало работает касс, в очереди стоят много народа. В магазине всегда чисто.'
